# Plagiarism Detection
In this assignment, we will sequentially practice the steps to build a plagiarism detection application using a pre-trained Word2Vec model.
Data: https://s3.amazonaws.com/video.udacity-data.com/topher/2019/January/5c4147f9_data/data.zip
This exercise requires knowledge of Python programming with the following libraries:
 * `gensim` (to load the Word2Vec model)
 * `numpy` (to compute similarity)
Additionally, we will use a pre-trained Word2Vec model `Google's pre-trained word2vec model`
Steps to Solve This Exercise
1. Exploring the Dataset  
2. Build a class for computing document similarity (`DocSim` class)
3. Create an instance of the above class (Load the pre-trained word embedding model, Create a list of stopwords and create an instance of the `DocSim` class)
4. Plagiarism Detection and Evaluation

In [7]:
# Import libraries
import pandas as pd
import numpy as np
import nltk
import matplotlib.pyplot as plt
import os, shutil
import requests
import zipfile
import datetime
print(f"This notebook was last run in: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

This notebook was last run in: 2025-10-01 08:17:46


## Load the dataset

In [ ]:
! cd data/ & wget -O 'local_filename.zip' 'https://s3.amazonaws.com/video.udacity-data.com/topher/2019/January/5c4147f9_data/data.zip'

In [8]:
data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)

zip_path = os.path.join(data_dir, "local_filename.zip")
print(zip_path)

def unzip(zip_path, data_dir, delete=True):
    if zip_path.endswith(".zip"):
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(data_dir)
        print("Extracted zip to", data_dir)
    else:
        print("This file format is not accepted.")
        return
    
    if delete:
        os.remove(zip_path)
        print("Removed!")
        return
    return

./data/local_filename.zip


In [ ]:
unzip(zip_path, data_dir)

### Load data_infor file

In [9]:
data_path = os.path.join(data_dir, "data")
file_info_path = os.path.join(data_path, "file_information.csv")
df = pd.read_csv(file_info_path)
print(df.head(5))

             File Task Category
0  g0pA_taska.txt    a      non
1  g0pA_taskb.txt    b      cut
2  g0pA_taskc.txt    c    light
3  g0pA_taskd.txt    d    heavy
4  g0pA_taske.txt    e      non


## Exflore the dataset

In [10]:
print("DATASET OVERVIEW")

print(f"\nTotal files: {len(df)}")
print(f"Categories: {df['Category'].unique()}")
print(f"\nCategory distribution:\n{df['Category'].value_counts()}")


DATASET OVERVIEW

Total files: 100
Categories: ['non' 'cut' 'light' 'heavy' 'orig']

Category distribution:
Category
non      38
cut      19
light    19
heavy    19
orig      5
Name: count, dtype: int64


## Build a DocSim class

In [12]:
from sklearn.metrics.pairwise import cosine_similarity
class DocSim:
    def __init__(self, model, stopwords=None):
        self.model = model
        self.stopwords = stopwords if stopwords else set()
    
    def preprocess(self, text):
        words = text.lower().split()
        return [w for w in words if w not in self.stopwords and w in self.model]
    
    def vectorize(self, text):
        words = self.preprocess(text)
        if not words:
            return np.zeros(self.model.vector_size)
        word_vectors = [self.model[w] for w in words]
        return np.mean(word_vectors, axis=0)
    
    def calculate_similarity(self, text1, text2):
        vec1 = self.vectorize(text1).reshape(1, -1)
        vec2 = self.vectorize(text2).reshape(1, -1)
        return cosine_similarity(vec1, vec2)[0][0]


### Load the Google's pre-trained word2vec model
We will use the pre-trained Word2Vec model provided by Google. This model was trained on a massive dataset of Google News articles and contains 300-dimensional vectors for 3 million words and phrases

In [ ]:
from gensim.models import KeyedVectors

model_dir = "./model"
model_path = f"{model_dir}/GoogleNews-vectors-negative300.bin/GoogleNews-vectors-negative300.bin"
print(f"Loading model...")
w2v_model = KeyedVectors.load_word2vec_format(model_path, binary=True)
print(f"Model loaded: {len(w2v_model)} words")

Model loaded: 3000000 words


In [13]:
from nltk.corpus import stopwords
import nltk

try:
    stop_words = set(stopwords.words('english'))
except LookupError:
    nltk.download('stopwords')
    stop_words = set(stopwords.words('english'))

print(f"Stopwords loaded: {len(stop_words)} words")
doc_sim = DocSim(w2v_model, stop_words)

Stopwords loaded: 198 words


## Starting detect plagiarism

In [14]:
def read_file(filepath):
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

In [15]:
from tqdm import tqdm
def detect_plagiarism_all(df, data_path, threshold=0.7):
    results = []
    files = df['File'].tolist()

    print("DETECTING PLAGIARISM")
    
    for i in tqdm(range(len(files)), desc="Processing files"):
        source_file = os.path.join(data_path, files[i])
        source_text = read_file(source_file)
        source_category = df.iloc[i]['Category']
        
        for j in range(i + 1, len(files)):
            target_file = os.path.join(data_path, files[j])
            target_text = read_file(target_file)
            target_category = df.iloc[j]['Category']
            
            similarity = doc_sim.calculate_similarity(source_text, target_text)
            
            results.append({
                'source_file': files[i],
                'target_file': files[j],
                'source_category': source_category,
                'target_category': target_category,
                'similarity': similarity,
                'plagiarism': similarity >= threshold
            })
    
    return pd.DataFrame(results)

In [16]:
results_df = detect_plagiarism_all(df, data_path, threshold=0.7)

DETECTING PLAGIARISM


Processing files: 100%|██████████| 100/100 [00:24<00:00,  4.11it/s]


## Evaluate the results

In [17]:
print("PLAGIARISM DETECTION RESULTS")

print(f"\nTotal comparisons: {len(results_df)}")
print(f"Plagiarism detected: {results_df['plagiarism'].sum()}")
print(f"Plagiarism rate: {results_df['plagiarism'].sum() / len(results_df) * 100:.2f}%")

print("\nTop 10 Most Similar Pairs")
top_similar = results_df.nlargest(10, 'similarity')
print(top_similar[['source_file', 'target_file', 'similarity', 'plagiarism']])

print("\nPlagiarism Cases (Similarity >= 0.7)")
plagiarism_cases = results_df[results_df['plagiarism'] == True]
print(plagiarism_cases[['source_file', 'target_file', 'similarity']])

PLAGIARISM DETECTION RESULTS

Total comparisons: 4950
Plagiarism detected: 3383
Plagiarism rate: 68.34%

Top 10 Most Similar Pairs
         source_file     target_file  similarity  plagiarism
1864  g0pE_taska.txt  orig_taska.txt    0.997990        True
4318  g3pA_taskd.txt  orig_taskd.txt    0.997770        True
4774  g4pC_taska.txt  orig_taska.txt    0.997105        True
4828  g4pC_taskd.txt  orig_taskd.txt    0.995707        True
1849  g0pE_taska.txt  g4pC_taska.txt    0.994758        True
4303  g3pA_taskd.txt  g4pC_taskd.txt    0.994729        True
3351  g2pA_taskc.txt  orig_taskc.txt    0.994128        True
761   g0pB_taskc.txt  orig_taskc.txt    0.990776        True
3638  g2pB_taskd.txt  g3pA_taskd.txt    0.990074        True
3673  g2pB_taskd.txt  orig_taskd.txt    0.989235        True

Plagiarism Cases (Similarity >= 0.7)
         source_file     target_file  similarity
1     g0pA_taska.txt  g0pA_taskc.txt    0.711468
2     g0pA_taska.txt  g0pA_taskd.txt    0.716860
3     g0pA_ta